In [3]:
import numpy as np
import cupy as cp
import gc 
import networkx as nx
import matplotlib.pyplot as plt
from simplicial_generator_class import SimplicialComplex

import time

from scipy.linalg import eigh # Ottimizzato per matrici Hermitiane/Simmetriche reali
from scipy.special import comb # Per calcolare il coefficiente binomiale


# LIBRERIE PER COMBINATORICA E PROGRESSO

from itertools import combinations # Fondamentale per i termini J_ijkl
from tqdm.notebook import tqdm     # Barra di caricamento perfetta per Colab
from scipy.stats import norm

# LIBRERIE PER PLOTTING E GRAFICA

import seaborn as sns # Eccellente per plottare le distribuzioni di Wigner-Dyson

# IMPOSTAZIONI GRAFICHE OPZIONALI

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "text.usetex": False, # Su Colab il LaTeX richiede installazioni extra, meglio False
    "font.family": "serif",
    "axes.labelsize": 12,
    "font.size": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

import mylib as ml

In [ ]:
# you can either initialize a stochastic simplicial complex governed by
# dimension related probabilities or with a particular chosen initialization (see below)
p_1 = 0
p_2 = 0
p_3 = 1
kwargs = {'initialization': 'tetrahedron', 'prob_new_link': p_1,
          'prob_new_triangle': p_2, 'prob_new_tetrahedron': p_3, }
N = 20
N_0 = 4 if kwargs['initialization'] == 'tetrahedron' else 3 if kwargs['initialization'] == 'triangle' \
                    else 2 if kwargs['initialization'] == 'link' else 1 if kwargs['initialization'] == 'node' else 0


N_increase = N-N_0

net_test = 500

N_iterazioni = 100


Gamma_tensor = ml.Dirac_gen_GPU(N)


r_est_list = np.empty(net_test, dtype=np.float64)
r_index_est_list = np.empty(net_test, dtype=np.float64)
sigma_est_list = np.empty(net_test, dtype=np.float64)
sigma_index_est_list = np.empty(net_test, dtype=np.float64)

max_bet_list = np.empty(net_test, dtype=np.float64)
degree_var_list = np.empty(net_test, dtype=np.float64)
N_com_list = np.empty(net_test, dtype=np.float64)
modularity_list = np.empty(net_test, dtype=np.float64)

for m in range(net_test):
    

    #Costruzione rete 
    simplex = SimplicialComplex(**kwargs)
    for i in range(N_increase): 
        simplex.add_simplex()
    
    
    
    
    G = nx.Graph()
    G.add_edges_from(simplex.links)


    edge_bet = nx.edge_betweenness_centrality(G)
    max_bet = np.max(list(edge_bet.values()))
    max_bet_list[m] = max_bet

    degree_var_list[m] = np.var(G.degree)

    com_greedy = nx.community.greedy_modularity_communities(G)
    N_com_list[m] = len(com_greedy)

    modularity_list[m] = nx.community.modularity(G, com_greedy)

    
    matrices_2, matrices_3, matrices_4 = ml.build_matrices_GPU(Gamma_tensor, simplex.links, simplex.triangles, simplex.tetrahedra)

    r_mean, r_index_mean, _, _ = ml.simp_stat_create_GPU(N_iterazioni, N, matrices_2, matrices_3, matrices_4)

    
    
    r_est = np.mean(r_mean)
    sigma_est = np.std(r_mean)/np.sqrt(len(r_mean))

    r_index_est = np.mean(r_index_mean)
    sigma_index_est = np.std(r_index_mean)/np.sqrt(len(r_index_mean))

    r_est_list[m] = r_est
    sigma_est_list[m] = sigma_est
    r_index_est_list[m] = r_index_est
    sigma_index_est_list[m] = sigma_index_est

    if 'matrices_2' in locals(): del matrices_2
    if 'matrices_3' in locals(): del matrices_3
    if 'matrices_4' in locals(): del matrices_4
    
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

    gc.collect()

    print(fr'Cycle index: {m}')

Cycle index: 0
Cycle index: 1
Cycle index: 2
Cycle index: 3
Cycle index: 4
Cycle index: 5
Cycle index: 6
Cycle index: 7
Cycle index: 8
Cycle index: 9


In [ ]:
"""

np.savez_compressed(
    'simulation_data_N16_500x100.npz', 
    variance= degree_var_list,
    betweenness = max_bet_list,
    r = r_est_list,
    sigma_r = sigma_est_list,
    r_tilde = r_index_est_list,
    sigma_r_tilde = sigma_index_est_list,
    N_com = N_com_list,
    modularity = modularity_list


)
"""


"\n\nnp.savez_compressed(\n    'dati_simulazione2_N16_500x100.npz', \n    variance= degree_var_list,\n    betweenness = max_bet_list,\n    r = r_est_list,\n    sigma_r = sigma_est_list,\n    r_tilde = r_index_est_list,\n    sigma_r_tilde = sigma_index_est_list,\n    N_com = N_com_list,\n    modularity = modularity_list\n\n\n)\n"